**Installation:**

In [ ]:
# 1. Installation
!pip install sentence-transformers gradio nltk -q

import gradio as gr
from sentence_transformers import SentenceTransformer, util
import torch
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Downloading essential NLTK data
nltk.download('punkt')
nltk.download('stopwords')

print("Setup Successful! NLP engine is ready.")

Setup Successful! NLP engine is ready.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Backend Logic (NLP and Similarity Matching):

In [ ]:
# Load the semantic model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Whole University FAQ Dataset
faq_data = {
    "What are the general admission requirements?": "Admission requirements vary by program, but generally, a minimum of 45-50% in your previous degree is required for undergraduate programs at University of Kotli.",
    "Does the university provide transport facilities?": "Yes, the university provides bus services for students and staff across Kotli and nearby designated routes.",
    "Are there separate hostels for girls and boys?": "Yes, University of Kotli offers separate and secure hostel facilities for both male and female students on a merit basis.",
    "How can I access the library?": "Students can access the central library using their valid University ID card during official working hours.",
    "What is the procedure for degree verification?": "For degree verification, students must submit an application form along with the prescribed fee at the Controller of Examinations office.",
    "Is there a sports complex on campus?": "Yes, the university encourages physical activities and provides facilities for cricket, football, and indoor games.",
    "Where is the main campus located?": "The main campus of University of Kotli Azad Jammu and Kashmir is located in Kotli, AJ&K.",
    "How do I apply for a character certificate?": "You can apply for a character certificate through your respective department's head office after clearing your dues.",
    "What are the official university timings?": "The university typically operates from 8:30 AM to 4:00 PM, Monday to Friday."
}

faq_questions = list(faq_data.keys())
faq_embeddings = model.encode(faq_questions, convert_to_tensor=True)

def university_bot_logic(user_query):
    # Professional Preprocessing
    user_query = user_query.lower().strip()
    user_query = re.sub(r'[^\w\s]', '', user_query)

    if len(user_query) < 3:
        return "I'm listening! Please ask a question about the university."

    # Intent Matching using Cosine Similarity
    query_embedding = model.encode(user_query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, faq_embeddings)[0]

    top_result_idx = torch.argmax(cos_scores)
    score = cos_scores[top_result_idx].item()

    # Accuracy Threshold
    if score < 0.40:
        return "I'm sorry, I couldn't find a direct answer. Please check the official university website or visit the Student Affairs office."

    return faq_data[faq_questions[top_result_idx]]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Professional Chat UI and Launch:**

In [ ]:
# Custom Soft Theme
theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
)

def respond(message, chat_history):
    bot_message = university_bot_logic(message)
    chat_history.append((message, bot_message))
    return "", chat_history

with gr.Blocks(theme=theme, title="UOK AJ&K Smart Support") as demo:
    gr.Markdown(
        """
        # 🏛️ **University of Kotli Smart Assistant**
        ### Official FAQ Support for Students & Visitors
        *Powered by Advanced Natural Language Processing*
        """
    )

    chatbot = gr.Chatbot(label="University Help Desk", height=450, bubble_full_width=False)

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask about Admissions, Hostels, Transport, or Exams...",
            show_label=False,
            scale=8
        )
        submit = gr.Button("Send", variant="primary", scale=2)

    gr.Examples(
        examples=["Does the university have hostels?", "Transport facility details", "How to verify degree?"],
        inputs=msg
    )

    gr.HTML(
        """
        <div style='text-align: center; border-top: 1px solid #ddd; padding-top: 10px; margin-top: 20px;'>
            <p>Developed by <b>Adeeba Malik</b> | AI Intern at CodeAlpha</p>
            <p><small>University of Kotli Azad Jammu and Kashmir</small></p>
        </div>
        """
    )

    submit.click(respond, [msg, chatbot], [msg, chatbot])
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

# Launching with error tracking
demo.launch(debug=True, share=True)

/tmp/ipykernel_18523/716715822.py:13: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, title="UOK AJ&K Smart Support") as demo:
/tmp/ipykernel_18523/716715822.py:22: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="University Help Desk", height=450, bubble_full_width=False)
/tmp/ipykernel_18523/716715822.py:22: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(label="University Help Desk", height=450, bubble_full_width=False)
/tmp/ipykernel_18523/716715822.py:22: Deprecati

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95261dc84a3188a8eb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
